<a href="https://colab.research.google.com/github/DaisyXUUUUU/CIS-5450-Project/blob/main/notebooks/01b_add_features_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1b — Add rush-hour, night, and bill-based tip columns (v2)

**Owner:** Ziyue Xu (Daisy)

This notebook does **not** download, sample, or clean anything. It reads the fixed file
`data/clean/trips_clean_v1.parquet` (796,989 rows × 48 columns, built by `01_data_pipeline.ipynb`),
adds six columns, checks that nothing else changed, and writes `trips_clean_v2.parquet` next to it.
v1 is left untouched. There is no randomness here, so every run gives the same v2.

| New column | Definition |
| --- | --- |
| `is_rush_hour` | 1 if pickup is Monday–Friday, 16:00–19:59, and not a federal holiday (TLC $2.50 rush surcharge rule, keyed on pickup time). |
| `is_night` | 1 if pickup is 20:00–05:59 (TLC $1 overnight surcharge window). |
| `tip_pct_bill` | `tip_amount / (total_amount - tip_amount)`, i.e. tip as a share of the bill before tip. Outcome, **not a feature**. |
| `high_tip_bill` | 1 if the tip is above 20% of the bill (see note in section 3). Alternative target, **not a feature**. |
| `high_tip_bill_25`, `high_tip_bill_30` | Same rule at 25% and 30% of the bill. Sensitivity checks only, **not features**. |

Columns in v1 that use the fare as denominator: `tip_pct`, `high_tip`, `high_tip_20`, `high_tip_30` (all kept unchanged), plus
cleaning rule 8 (`tip / fare <= 100%`, which removed 1,205 rows). No model feature uses the fare as a denominator.

Federal holidays in the data window (April 1 – June 30, 2026): **Memorial Day, Monday May 25, 2026** (last Monday of May)
and **Juneteenth, Friday June 19, 2026**. There are no other federal holidays between April and June
(Presidents' Day is in February, Independence Day is July 4).

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_DIR = Path(os.environ.get("CIS5450_DATA_DIR", "/content/drive/MyDrive/CIS5450_project/data"))
CLEAN_DIR = DATA_DIR / "clean"
V1_PATH = CLEAN_DIR / "trips_clean_v1.parquet"
V2_PATH = CLEAN_DIR / "trips_clean_v2.parquet"
assert V1_PATH.exists(), f"v1 file not found: {V1_PATH}"

EXPECTED_ROWS, EXPECTED_COLS = 796_989, 48
print("v1:", V1_PATH)
print("v2:", V2_PATH)

## 1. Load v1 (read only)

In [ ]:
v1 = pd.read_parquet(V1_PATH)
print(v1.shape)
assert v1.shape == (EXPECTED_ROWS, EXPECTED_COLS), f"v1 shape is {v1.shape}, expected {(EXPECTED_ROWS, EXPECTED_COLS)}"

ORIG_COLS = list(v1.columns)
NEW_COLS = ["is_rush_hour", "is_night", "tip_pct_bill", "high_tip_bill", "high_tip_bill_25", "high_tip_bill_30"]
assert not set(NEW_COLS) & set(ORIG_COLS), "a new column name already exists in v1"

# Fingerprint of v1 so we can prove later that no original value changed.
v1_hash = pd.util.hash_pandas_object(v1, index=True).sum()
print("v1 fingerprint:", v1_hash)
print("pickup range:", v1["tpep_pickup_datetime"].min(), "to", v1["tpep_pickup_datetime"].max())
print(v1["split"].value_counts().reindex(["train", "val", "test"]))

## 2. Time flags: `is_rush_hour` and `is_night`

In [ ]:
FEDERAL_HOLIDAYS_2026 = pd.to_datetime(["2026-05-25",   # Memorial Day (last Monday of May)
                                        "2026-06-19"])  # Juneteenth
assert pd.Timestamp("2026-05-25").day_name() == "Monday"
assert pd.Timestamp("2026-05-25") + pd.Timedelta(days=7) > pd.Timestamp("2026-05-31")  # it is the LAST Monday
assert pd.Timestamp("2026-06-19").day_name() == "Friday"

df = v1.copy()
pu = df["tpep_pickup_datetime"]
hour, dow, day = pu.dt.hour, pu.dt.dayofweek, pu.dt.normalize()
is_holiday = day.isin(FEDERAL_HOLIDAYS_2026)

df["is_rush_hour"] = ((dow <= 4) & hour.between(16, 19) & ~is_holiday).astype("int8")
df["is_night"] = ((hour >= 20) | (hour <= 5)).astype("int8")

print("trips on a federal holiday:", int(is_holiday.sum()))
print("weekday 16-19h trips that are on a holiday (so NOT rush):", int(((dow <= 4) & hour.between(16, 19) & is_holiday).sum()))
print("is_rush_hour share:", round(df["is_rush_hour"].mean(), 4))
print("is_night share:    ", round(df["is_night"].mean(), 4))
pd.crosstab(hour, [df["is_rush_hour"], df["is_night"]]).head(24)

**Sanity check against the meter.** For Vendor 2 (Curb), `extra` holds only the rush and overnight surcharges,
so rush-hour trips should mostly show `extra >= 2.5` and other daytime trips should not. Vendor 1 is left out
because its `extra` also bundles the congestion, CBD, and airport fees.

In [ ]:
v2_only = df[df["VendorID"] == 2]
check = (v2_only.assign(extra_ge_2_5=(v2_only["extra"] >= 2.5).astype(int))
                .groupby(["is_rush_hour", "is_night"])["extra_ge_2_5"].agg(["mean", "size"]))
print(check)
# Holidays: the meter should NOT charge the rush surcharge on May 25 / June 19, 16-19h.
hol = v2_only[v2_only["tpep_pickup_datetime"].dt.normalize().isin(FEDERAL_HOLIDAYS_2026)
              & v2_only["tpep_pickup_datetime"].dt.hour.between(16, 19)]
print("Holiday 16-19h Vendor 2 trips:", len(hol), "| share with extra >= 2.5:", round((hol["extra"] >= 2.5).mean(), 3))

## 3. Bill-based tip: `tip_pct_bill` and `high_tip_bill`

`bill = total_amount - tip_amount` is everything the rider pays before the tip (fare plus all surcharges, taxes, and tolls).
This avoids the Vendor 1 `extra` double counting, because it does not add up the components.

About half of all trips tip exactly 20% of the bill (the screen preset), but the tip is rounded to the cent,
so `tip / bill` lands a hair above or below 0.20 for reasons that have nothing to do with the rider.
To keep those preset trips on the same side of the line, `high_tip_bill = 1` only if the tip is more than
**one cent above** 20% of the bill, i.e. `tip_amount > round(0.20 * bill, 2) + 0.005`.
The 25% and 30% sensitivity labels use the same one-cent rule. A naive `tip_pct_bill > 0.20` is also printed for comparison but not saved.

In [ ]:
bill = (df["total_amount"] - df["tip_amount"]).round(2)
print("rows with bill <= 0:", int((bill <= 0).sum()))

df["tip_pct_bill"] = np.where(bill > 0, df["tip_amount"] / bill, np.nan)
df["high_tip_bill"] = (df["tip_amount"] > (0.20 * bill).round(2) + 0.005).astype("int8")
df.loc[bill <= 0, "high_tip_bill"] = 0
# Sensitivity labels on the bill, same one-cent rule (mirror high_tip_20 / high_tip_30 in v1).
for t in (0.25, 0.30):
    col = f"high_tip_bill_{int(t * 100)}"
    df[col] = (df["tip_amount"] > (t * bill).round(2) + 0.015).astype("int8")
    df.loc[bill <= 0, col] = 0

# v1 cleaning rule 8 dropped tip > fare. Since bill >= fare for almost all trips, tip_pct_bill <= 1 should hold already.
print("rows with bill < fare:", int((bill < df["fare_amount"]).sum()))
print("rows with tip_pct_bill > 1:", int((df["tip_pct_bill"] > 1).sum()), "| max:", round(df["tip_pct_bill"].max(), 4))

at_20 = (df["tip_amount"] - (0.20 * bill).round(2)).abs() <= 0.01
print("share of trips tipping 20% of bill (within 1 cent):", round(at_20.mean(), 4))
print("tip_pct_bill median:", round(df["tip_pct_bill"].median(), 4))
summary = pd.DataFrame({
    "high_tip_bill (saved)": df.groupby("split")["high_tip_bill"].mean(),
    "naive tip_pct_bill > 0.20": (df["tip_pct_bill"] > 0.20).groupby(df["split"]).mean(),
    "high_tip_bill_25": df.groupby("split")["high_tip_bill_25"].mean(),
    "high_tip_bill_30": df.groupby("split")["high_tip_bill_30"].mean(),
    "high_tip (v1, fare >= 25%)": df.groupby("split")["high_tip"].mean(),
}).reindex(["train", "val", "test"]).round(4)
summary

## 4. Checks: same rows, original columns unchanged

In [ ]:
assert len(df) == EXPECTED_ROWS, len(df)
assert list(df.columns) == ORIG_COLS + NEW_COLS
assert df[ORIG_COLS].equals(v1), "an original column changed"
assert pd.util.hash_pandas_object(df[ORIG_COLS], index=True).sum() == v1_hash
assert df[["is_rush_hour", "is_night", "high_tip_bill", "high_tip_bill_25", "high_tip_bill_30"]].isin([0, 1]).all().all()
assert not (df["is_rush_hour"].astype(bool) & df["is_night"].astype(bool)).any(), "rush and night overlap"
print("All checks passed:", df.shape)

## 5. Write v2 (v1 is kept) and re-read it

In [ ]:
df.to_parquet(V2_PATH, index=False)
back = pd.read_parquet(V2_PATH)
assert back.shape == (EXPECTED_ROWS, EXPECTED_COLS + len(NEW_COLS)), back.shape
assert back[ORIG_COLS].reset_index(drop=True).equals(v1.reset_index(drop=True))
assert pd.read_parquet(V1_PATH).shape == (EXPECTED_ROWS, EXPECTED_COLS)  # v1 still intact
print("Wrote", V2_PATH, back.shape, f"{V2_PATH.stat().st_size / 1e6:.1f} MB")
back[NEW_COLS].describe().T